# NIMBO — Image Routing Benchmark v2 (T4)
**16 prompts × 3 paths:** Always Fast (LCM), Always Standard (SD 1.5), and NIMBO (fast → CLIP check → standard fallback).

- Real GPU inference; all model calls use the same prompt and seed per item.
- **Threshold 0.65 is provisional** and is not calibrated; do not claim measured decision accuracy until blinded human labels are completed.
- Standard-only inference is generated for **every** prompt, including those accepted by NIMBO, to enable matched quality comparisons.
- Results exclude initial model loading and warmups; loading durations are reported separately. Sequential loading means this is **not production deployment latency**.
- CLIP contrasts are *relative*, not calibrated presence probabilities. Safety checker is disabled in this research notebook: do not expose as a public generation service.


In [ ]:
# Safe dependency setup (torchao is not required by this notebook)
%pip -q install "diffusers>=0.30,<1" "transformers>=4.51,<5" "accelerate>=0.30" "peft>=0.12" "safetensors>=0.4" pandas pillow
%pip -q uninstall -y torchao
# If you removed torchao for the first time in this runtime, restart session once,
# then rerun cells from the top. Do not install torchao in later cells.


In [ ]:
import gc, json, math, time
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from diffusers import StableDiffusionPipeline, LCMScheduler
assert torch.cuda.is_available(), "Enable T4 from Runtime > Change runtime type."
print('GPU:',torch.cuda.get_device_name(0))
BASE='stable-diffusion-v1-5/stable-diffusion-v1-5'
LORA='latent-consistency/lcm-lora-sdv1-5'
OUTPUT=Path('/content/nimbo_image_v2_results');OUTPUT.mkdir(parents=True,exist_ok=True)
THRESHOLD=0.65  # fixed BEFORE looking at human labels: exploratory, NOT calibrated
PROMPTS=[
 ('single red apple centered on white studio background, product photo','apple','simple'),
 ('blue ceramic mug on a wooden table, studio photography','mug','simple'),
 ('a single yellow sunflower against a pale sky, photorealistic','sunflower','simple'),
 ('a brown leather backpack on a wooden bench, product photo','backpack','simple'),
 ('minimal white ceramic teapot on a wooden table, studio photograph','teapot','simple'),
 ('a black bicycle leaning against a brick wall, realistic photograph','bicycle','simple'),
 ('a small red sailboat on a calm lake at sunset, sailboat clearly visible','sailboat','object_critical'),
 ('a bright orange lifebuoy hanging on a boat railing at sea, realistic photo','lifebuoy','object_critical'),
 ('a yellow taxi driving down a busy rainy street at night, realistic photograph','taxi','object_critical'),
 ('an astronaut standing on the moon holding a blue flag, photorealistic','flag','object_critical'),
 ('sunlit library room with shelves full of books and a reading chair, architecture photo','books','scene'),
 ('wind turbines above a green farm with mountains in the background, photograph','wind turbines','scene'),
 ('a train crossing a bridge over a river at sunrise, photograph','train','scene'),
 ('a red umbrella in the foreground of a crowded city square, photograph','umbrella','scene'),
 ('a glass greenhouse containing tomato plants next to a farmhouse, photograph','greenhouse','complex'),
 ('a small white dog wearing a blue scarf sitting beside a red bicycle, photograph','dog','complex'),
]
SEEDS=[17+13*i for i in range(len(PROMPTS))]
print('Prompts:',len(PROMPTS),'threshold:',THRESHOLD)


In [ ]:
# Load checker and warm it up once. Its time is measured per item below.
clip=CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to('cuda').eval()
processor=CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
def assess(image,prompt,required_object):
    captions=[prompt, 'an unrelated abstract geometric pattern',
              f'a photograph with a clearly visible {required_object}',
              f'a photograph with no {required_object} visible']
    inputs=processor(text=captions,images=image,return_tensors='pt',padding=True,truncation=True).to('cuda')
    torch.cuda.synchronize();start=time.perf_counter()
    with torch.inference_mode():
        logits=clip(**inputs).logits_per_image[0]
        alignment=logits[:2].softmax(dim=-1)[0].item()
        object_score=logits[2:].softmax(dim=-1)[0].item()
    torch.cuda.synchronize()
    check_s=time.perf_counter()-start
    return {'alignment':round(alignment,5),'object_relative_score':round(object_score,5),
            'checker_s':round(check_s,5),'passed':bool(alignment>=THRESHOLD and object_score>=THRESHOLD)}


In [ ]:
def build_pipe(mode):
    p=StableDiffusionPipeline.from_pretrained(BASE,torch_dtype=torch.float16,safety_checker=None).to('cuda')
    p.set_progress_bar_config(disable=True)
    if mode=='lcm':
        p.load_lora_weights(LORA)
        p.scheduler=LCMScheduler.from_config(p.scheduler.config)
    return p

def render(p,mode,prompt,seed):
    # CLIP occupies GPU in parallel with generator. Keep batch size 1.
    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();start=time.perf_counter()
    with torch.inference_mode():
        image=p(prompt,width=512,height=512,
                num_inference_steps=4 if mode=='lcm' else 25,
                guidance_scale=1.0 if mode=='lcm' else 7.5,
                generator=torch.Generator(device='cuda').manual_seed(seed)).images[0]
    torch.cuda.synchronize()
    return image, time.perf_counter()-start,torch.cuda.max_memory_allocated()/1e9

records=[]
start=time.perf_counter();fast=build_pipe('lcm');fast_load_s=time.perf_counter()-start
_ = render(fast,'lcm',PROMPTS[0][0],SEEDS[0])  # warmup excluded
for i,(prompt,obj,category) in enumerate(PROMPTS):
    image,lat,peak=render(fast,'lcm',prompt,SEEDS[i])
    quality=assess(image,prompt,obj)
    image.save(OUTPUT/f'fast_{i:02d}.png')
    records.append({'prompt_id':i,'prompt':prompt,'required_object':obj,'category':category,
                    'seed':SEEDS[i],'fast_s':round(lat,5),'fast_peak_allocated_gb':round(peak,4),
                    **quality,'route':'fast_accepted' if quality['passed'] else 'escalated_to_standard'})
    print(f'Fast {i+1}/{len(PROMPTS)}: {records[-1]["route"]} object_score={quality["object_relative_score"]:.3f}')
del fast;gc.collect();torch.cuda.empty_cache()
print('LCM pipeline load (excluded):',round(fast_load_s,2),'s')


In [ ]:
# Always-Standard reference for ALL prompts; also supplies NIMBO's fallback image.
start=time.perf_counter();standard=build_pipe('standard');standard_load_s=time.perf_counter()-start
_ = render(standard,'standard',PROMPTS[0][0],SEEDS[0]) # warmup excluded
for r in records:
    i=r['prompt_id'];image,lat,peak=render(standard,'standard',r['prompt'],r['seed'])
    image.save(OUTPUT/f'standard_{i:02d}.png')
    r['standard_s']=round(lat,5);r['standard_peak_allocated_gb']=round(peak,4)
    r['nimbo_s']=round(r['fast_s']+r['checker_s']+(lat if not r['passed'] else 0),5)
    r['nimbo_image']=f'fast_{i:02d}.png' if r['passed'] else f'standard_{i:02d}.png'
    r['human_fast_object_present']=''   # label later: 1 or 0
    r['human_standard_object_present']='' # label later: 1 or 0
    print('Standard reference',i+1,'/',len(records))
del standard;gc.collect();torch.cuda.empty_cache()


In [ ]:
# Report pipeline inference only. Does not include loading or transfers.
df=pd.DataFrame(records)
lat_fast=df.fast_s.mean();lat_large=df.standard_s.mean();lat_nimbo=df.nimbo_s.mean()
report={'gpu':torch.cuda.get_device_name(0),'base_model':BASE,'lora':LORA,
        'prompts':len(records),'threshold_provisional':THRESHOLD,
        'fallback_count':int((~df.passed).sum()),'fallback_rate':round((~df.passed).mean(),4),
        'always_fast_mean_s':round(lat_fast,4),'always_standard_mean_s':round(lat_large,4),
        'nimbo_mean_s':round(lat_nimbo,4),'nimbo_p95_s':round(float(df.nimbo_s.quantile(.95)),4),
        'nimbo_latency_reduction_vs_standard_pct':round(100*(1-lat_nimbo/lat_large),2),
        'total_checker_s':round(df.checker_s.sum(),3),
        'model_load_s_excluded':{'lcm':round(fast_load_s,2),'standard':round(standard_load_s,2)},
        'quality_validated':False,
        'limitations':['Threshold 0.65 was not calibrated; human labels needed.',
                       'CLIP relative scores are not object presence probabilities.',
                       'Not live interleaved inference: standard and fast models loaded sequentially.',
                       'Model loading and warmup excluded; switching costs may be higher in deployment.',
                       'The same fixed seeds do not force identical visual content across schedulers.',
                       'Peak PyTorch allocated memory is not total process GPU memory.',
                       'Safety checker disabled for offline research only.',
                       'No energy, FLOPs, dollars or production throughput measured.']}
(OUTPUT/'benchmark_v2.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
df.to_csv(OUTPUT/'benchmark_v2_raw.csv',index=False)
# Human review sheet, deliberately blank; human labels MUST NOT be inferred from CLIP.
df[['prompt_id','category','prompt','required_object','route','human_fast_object_present','human_standard_object_present']].to_csv(OUTPUT/'human_review_v2.csv',index=False)
print(json.dumps(report,indent=2))
display(df[['prompt_id','category','route','object_relative_score','fast_s','checker_s','standard_s','nimbo_s']])


In [ ]:
# Preview image pairs before labeling: Fast on left, Standard on right.
from IPython.display import display
from PIL import ImageDraw
for i in range(len(PROMPTS)):
    left=Image.open(OUTPUT/f'fast_{i:02d}.png').convert('RGB')
    right=Image.open(OUTPUT/f'standard_{i:02d}.png').convert('RGB')
    canvas=Image.new('RGB',(1024,540),'white')
    canvas.paste(left,(0,28));canvas.paste(right,(512,28))
    ImageDraw.Draw(canvas).text((10,6),f'{i}: LCM (fast)',fill='black')
    ImageDraw.Draw(canvas).text((522,6),f'{i}: Standard',fill='black')
    canvas.save(OUTPUT/f'compare_{i:02d}.jpg',quality=85)
    display(canvas.resize((768,405)))
print('Mark human_review_v2.csv: 1 = clearly visible, 0 = absent/unclear; inspect both sides without referring to CLIP prediction.')


In [ ]:
# Archive for download; the review CSV is still unlabeled until you edit it.
import shutil
archive=shutil.make_archive('/content/nimbo_image_v2_results','zip','/content','nimbo_image_v2_results')
print('Download:',archive)
from google.colab import files
files.download(archive)
